# Compilación Cuántica con t|ket⟩ y Qiskit

**Laboratorio 33 · Nivel muy avanzado**

Este laboratorio compara estrategias de compilación de Qiskit vs t|ket⟩ (Cambridge
Quantum/Quantinuum). Ambos compiladores transforman circuitos lógicos en circuitos
ISA (hardware-específicos), minimizando puertas de 2 qubits y profundidad.

> **Dependencia opcional:** `pip install pytket pytket-qiskit` para usar t|ket⟩ directamente.
> Sin ella, el laboratorio implementa las métricas de compilación con Qiskit.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import time

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Operator, process_fidelity
from qiskit_aer import AerSimulator
from qiskit.circuit.library import QFT, GroverOperator
from qiskit.transpiler import CouplingMap
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Intentar importar pytket
try:
    import pytket
    from pytket.extensions.qiskit import qiskit_to_tk, tk_to_qiskit
    from pytket.passes import FullPeepholeOptimise, SequencePass, KAKDecomposition
    HAS_TKET = True
    print(f'pytket {pytket.__version__} disponible')
except ImportError:
    HAS_TKET = False
    print('pytket no instalado — usando solo Qiskit (instala con: pip install pytket pytket-qiskit)')

print('Qiskit transpiler listo')


## 1. Circuitos de referencia

Definimos 4 circuitos de benchmark con características distintas:
- **QFT**: alta conectividad, muchas rotaciones
- **Grover 4q**: oráculo + difusor, puertas CCX
- **QAOA**: estructura de grafo
- **VQE ansatz**: capas RY+CX repetidas

In [ ]:
def circuito_grover_4q() -> QuantumCircuit:
    """Grover 2 iteraciones, target |1010>."""
    n = 4
    qc = QuantumCircuit(n)
    qc.h(range(n))
    for _ in range(2):
        # Oráculo |1010>
        qc.x([1, 3])
        qc.h(3); qc.ccx(0,1,3); qc.h(3)  # simplificación del MCX
        qc.x([1, 3])
        # Difusor
        qc.h(range(n)); qc.x(range(n))
        qc.h(3); qc.ccx(0,1,3); qc.h(3)
        qc.x(range(n)); qc.h(range(n))
    return qc

def circuito_qaoa_4q(gamma=0.7, beta=0.4) -> QuantumCircuit:
    """QAOA p=2 en grafo cuadrado 0-1-2-3-0."""
    qc = QuantumCircuit(4)
    qc.h(range(4))
    for _ in range(2):
        for i, j in [(0,1),(1,2),(2,3),(3,0)]:
            qc.cx(i,j); qc.rz(2*gamma,j); qc.cx(i,j)
        qc.rx(2*beta, range(4))
    return qc

def circuito_vqe_ansatz(n_capas=3) -> QuantumCircuit:
    """HWE ansatz 4q, n_capas RY+CX."""
    from qiskit.circuit import ParameterVector
    n = 4
    theta = ParameterVector('θ', n*(n_capas+1))
    qc = QuantumCircuit(n)
    idx = 0
    for q in range(n):
        qc.ry(theta[idx], q); idx += 1
    for _ in range(n_capas):
        for q in range(n-1): qc.cx(q,q+1)
        for q in range(n):
            qc.ry(theta[idx], q); idx += 1
    return qc

CIRCUITOS = {
    'QFT 4q':    QFT(4, do_swaps=False),
    'Grover 4q': circuito_grover_4q(),
    'QAOA p=2':  circuito_qaoa_4q(),
    'VQE HWE':   circuito_vqe_ansatz(3),
}

print('Circuitos de referencia:')
for nombre, qc in CIRCUITOS.items():
    print(f'  {nombre:12s}: {qc.num_qubits}q | '
          f'profundidad={qc.depth()} | '
          f'puertas={qc.size()} | '
          f'2q-gates={sum(1 for _,_,_ in qc.data if qc._data[_].operation.num_qubits==2 if hasattr(qc._data[_].operation,"num_qubits")) if False else len([g for g in qc.data if g.operation.num_qubits==2])}')


## 2. Niveles de optimización de Qiskit

Qiskit tiene 4 niveles de optimización (`optimization_level` 0-3):
- **0**: sin optimización — solo asignación básica
- **1**: simplificación leve (cancelación de puertas adyacentes)
- **2**: optimización moderada (peephole, KAK)
- **3**: optimización agresiva (síntesis Solovay-Kitaev, búsqueda de layout)

In [ ]:
# Topología: heavy-hex de 5 qubits (similar a IBM Falcon)
coupling_map = CouplingMap([[0,1],[1,2],[2,3],[3,4],[1,3]])
BASIS_GATES = ['cx','rz','sx','x','id']

resultados_qiskit = {}

for nombre, qc in CIRCUITOS.items():
    resultados_qiskit[nombre] = {}
    for level in range(4):
        t0 = time.perf_counter()
        qc_t = transpile(
            qc, coupling_map=coupling_map,
            basis_gates=BASIS_GATES,
            optimization_level=level,
            seed_transpiler=42
        )
        elapsed = time.perf_counter() - t0
        cx_count = sum(1 for g in qc_t.data if g.operation.name == 'cx')
        resultados_qiskit[nombre][level] = {
            'depth': qc_t.depth(),
            'cx':    cx_count,
            'gates': qc_t.size(),
            'time_ms': elapsed * 1000
        }

print(f'{"Circuito":12s} | {"Lvl":>3} | {"Depth":>6} | {"CX":>4} | {"Gates":>6} | {"t(ms)":>7}')
print('-' * 55)
for nombre in CIRCUITOS:
    for level, r in resultados_qiskit[nombre].items():
        print(f'{nombre:12s} | {level:>3} | {r["depth"]:>6} | {r["cx"]:>4} | {r["gates"]:>6} | {r["time_ms"]:>7.1f}')


In [ ]:
# Visualización: reducción de CX gates por nivel de optimización
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colores = ['#e74c3c','#f39c12','#2ecc71','#3498db']
nombres = list(CIRCUITOS.keys())
x = np.arange(len(nombres))
width = 0.2

for level in range(4):
    cx_vals  = [resultados_qiskit[n][level]['cx']    for n in nombres]
    dep_vals = [resultados_qiskit[n][level]['depth'] for n in nombres]
    axes[0].bar(x + level*width - 1.5*width, cx_vals,  width, label=f'Nivel {level}', color=colores[level], alpha=0.85)
    axes[1].bar(x + level*width - 1.5*width, dep_vals, width, label=f'Nivel {level}', color=colores[level], alpha=0.85)

for ax, title, ylabel in zip(axes, ['Puertas CX tras compilación', 'Profundidad tras compilación'], ['Número de CX', 'Profundidad']):
    ax.set_xticks(x); ax.set_xticklabels(nombres, rotation=15, ha='right')
    ax.set_title(title); ax.set_ylabel(ylabel)
    ax.legend(fontsize=8); ax.grid(alpha=0.3, axis='y')

plt.suptitle('Efecto del nivel de optimización de Qiskit', fontsize=12)
plt.tight_layout(); plt.show()


## 3. t|ket⟩: passes de optimización (si disponible)

t|ket⟩ organiza la optimización como una secuencia de *passes*:
- `FullPeepholeOptimise`: reescritura local de ventana deslizante
- `KAKDecomposition`: descomposición KAK de puertas de 2 qubits
- `CliffordSimp`: simplificación de subcircuitos Clifford

In [ ]:
if HAS_TKET:
    from pytket.passes import FullPeepholeOptimise, DecomposeBoxes
    from pytket.architecture import Architecture

    arch = Architecture([(0,1),(1,2),(2,3),(3,4),(1,3)])
    resultados_tket = {}

    for nombre, qc in CIRCUITOS.items():
        try:
            circ_tk = qiskit_to_tk(qc)
            DecomposeBoxes().apply(circ_tk)
            FullPeepholeOptimise().apply(circ_tk)
            qc_back = tk_to_qiskit(circ_tk)
            cx_tket = sum(1 for g in qc_back.data if g.operation.name in ('cx','cnot'))
            resultados_tket[nombre] = {'depth': qc_back.depth(), 'cx': cx_tket}
        except Exception as e:
            resultados_tket[nombre] = {'depth': None, 'cx': None, 'error': str(e)}

    print('Resultados t|ket⟩ vs Qiskit nivel 3:')
    print(f'{"Circuito":12s} | {"Qiskit CX":>10} | {"tket CX":>8} | {"Mejora":>7}')
    print('-' * 45)
    for nombre in CIRCUITOS:
        q3 = resultados_qiskit[nombre][3]['cx']
        tk = resultados_tket[nombre]['cx']
        mejora = f'{(q3-tk)/q3*100:.1f}%' if tk is not None and q3 > 0 else 'N/A'
        print(f'{nombre:12s} | {q3:>10} | {str(tk):>8} | {mejora:>7}')
else:
    print('pytket no disponible. Comparativa simulada con distintos passes de Qiskit:')
    print()

    # Comparativa usando PM de Qiskit con distintas estrategias de routing
    for nombre, qc in list(CIRCUITOS.items())[:2]:
        for routing in ['sabre', 'stochastic']:
            qc_t = transpile(qc, coupling_map=coupling_map, basis_gates=BASIS_GATES,
                             routing_method=routing, optimization_level=3, seed_transpiler=42)
            cx = sum(1 for g in qc_t.data if g.operation.name == 'cx')
            print(f'  {nombre} | routing={routing:11s} → CX={cx}, depth={qc_t.depth()}')


## 4. Análisis: fidelidad de proceso vs overhead de compilación

Para un modelo de ruido realista, más puertas no equivale siempre a menor fidelidad
si el routing introduce SWAPs mal colocados.

In [ ]:
from qiskit_aer.noise import NoiseModel, depolarizing_error

# Modelo de ruido con error CX del 0.5%
nm = NoiseModel()
nm.add_all_qubit_quantum_error(depolarizing_error(0.001, 1), ['sx','rz','x'])
nm.add_all_qubit_quantum_error(depolarizing_error(0.005, 2), ['cx'])

sim_noisy = AerSimulator(noise_model=nm)
SHOTS = 2048

# Benchmark: QFT 4q ideal vs compilado con distintos niveles
qc_bench = QFT(4, do_swaps=False)
qc_bench.measure_all()

fidelidades = {}

# Distribución ideal
sim_ideal = AerSimulator()
qc_ideal_t = transpile(qc_bench, sim_ideal, optimization_level=3, seed_transpiler=42)
counts_ideal = sim_ideal.run(qc_ideal_t, shots=SHOTS).result().get_counts()
n_estados = 2**4
estados = [format(i,'04b') for i in range(n_estados)]
probs_ideal = np.array([counts_ideal.get(s,0)/SHOTS for s in estados])

for level in range(4):
    qc_t = transpile(qc_bench, sim_noisy, basis_gates=BASIS_GATES,
                     coupling_map=coupling_map, optimization_level=level,
                     seed_transpiler=42)
    counts_n = sim_noisy.run(qc_t, shots=SHOTS).result().get_counts()
    probs_n = np.array([counts_n.get(s,0)/SHOTS for s in estados])
    tvd = 0.5 * np.sum(np.abs(probs_ideal - probs_n))
    cx  = sum(1 for g in qc_t.data if g.operation.name=='cx')
    fidelidades[level] = {'tvd': tvd, 'cx': cx, 'depth': qc_t.depth()}
    print(f'Nivel {level}: CX={cx:3d} | depth={qc_t.depth():3d} | TVD={tvd:.4f}')

# Figura
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
levels = list(fidelidades.keys())
tvds   = [fidelidades[l]['tvd'] for l in levels]
cxs    = [fidelidades[l]['cx']  for l in levels]

axes[0].bar(levels, tvds, color='#e74c3c', alpha=0.8)
axes[0].set_xlabel('Nivel de optimización'); axes[0].set_ylabel('TVD (↓ mejor)')
axes[0].set_title('Fidelidad vs nivel optimización (QFT 4q)'); axes[0].grid(alpha=0.3,axis='y')

axes[1].scatter(cxs, tvds, c=levels, cmap='RdYlGn_r', s=120, zorder=5)
for l in levels:
    axes[1].annotate(f'Nv{l}', (fidelidades[l]['cx'], fidelidades[l]['tvd']),
                     textcoords='offset points', xytext=(5,5), fontsize=9)
axes[1].set_xlabel('Número de CX gates'); axes[1].set_ylabel('TVD')
axes[1].set_title('TVD vs #CX (trade-off compilación)'); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()


## 5. Síntesis de unitarias: KAK y Solovay-Kitaev

Toda puerta de 2 qubits puede descomponerse en máximo **3 CX** + rotaciones
de 1 qubit (descomposición KAK). Solovay-Kitaev aproxima cualquier unitaria
en una base discreta (T, H, S) con profundidad $O(\log^c(1/\varepsilon))$.

In [ ]:
# Demostración: síntesis de una unitaria de 2 qubits aleatoria
from qiskit.synthesis import TwoQubitBasisDecomposer
from qiskit.circuit.library import CXGate

decomposer = TwoQubitBasisDecomposer(CXGate())

# Generar unitaria aleatoria de 2 qubits (haar-random)
rng = np.random.default_rng(7)
from scipy.stats import unitary_group
U = unitary_group.rvs(4, random_state=rng)

qc_sintetizado = decomposer(U)
cx_count = sum(1 for g in qc_sintetizado.data if g.operation.name == 'cx')

print(f'Unitaria aleatoria 2q sintetizada:')
print(f'  CX gates: {cx_count} (máximo teórico: 3)')
print(f'  Profundidad: {qc_sintetizado.depth()}')
print(f'  Total puertas: {qc_sintetizado.size()}')
print()

# Verificar fidelidad de proceso
U_sintetizado = Operator(qc_sintetizado).data
fidelidad = abs(np.trace(U.conj().T @ U_sintetizado)) / 4
print(f'Fidelidad de proceso = {fidelidad:.6f} (1.0 = perfecta)')
print(qc_sintetizado.draw('text'))


## 6. Resumen: qué compilador usar

| Situación | Recomendación |
|---|---|
| Prototipado rápido | Qiskit nivel 1 |
| Ejecución en hardware IBM | Qiskit nivel 3 + SABRE routing |
| Circuitos grandes (>50q) | t\|ket⟩ FullPeepholeOptimise |
| Hardware Quantinuum/IonQ | t\|ket⟩ nativo |
| Unitaria arbitraria 2q | KAK (≤3 CX) |
| Base discreta Clifford+T | Solovay-Kitaev |
